# Reinforcement Learning for Language Models

A language model is a causal transformer trained to predict the next token of a text. Trained on enough text, it produces fluent continuations of any prompt; but *fluent* is not the same as *helpful*, *harmless*, or *correct*, and the pretraining objective has no notion of any of these. The step that turns a text predictor into an assistant is, today, a reinforcement learning step: the model is treated as a **policy** that produces a response to a prompt, a **reward** judges the response, and the policy is improved by the algorithms of the previous chapters, with PPO in a central role. This is the technique known as [**reinforcement learning from human feedback**](./papers/2022%20-%20Training%20Language%20Models%20to%20Follow%20Instructions%20with%20Human%20Feedback.pdf) (RLHF), born in the RL literature as [learning from human preferences](./papers/2017%20-%20Deep%20Reinforcement%20Learning%20from%20Human%20Preferences.pdf) before being [applied to language models](./papers/2019%20-%20Fine-Tuning%20Language%20Models%20from%20Human%20Preferences.pdf), and the more recent methods for training models to reason, which reward correct answers to problems with verifiable solutions, are built on the same foundations.

This chapter applies the whole course to this problem. The language model is an agent whose actions are tokens; its episodes are responses; the reward arrives at the end of the response, from a **reward model** trained on human comparisons or from an automatic verifier; and the policy gradient methods of Chapters 08, 09 and 11 do the learning, with one essential addition, a **penalty on the divergence from the initial model**, which Chapter 11 has prepared us to understand. We then see how the same objective can be optimised without reinforcement learning at all, by **Direct Preference Optimization**, and how a simplified policy gradient, **GRPO**, is used with verifiable rewards.

The chapter is built around two toy worlds small enough to train in a notebook, with the transformer of Chapter 12: a language of short sentences, on which we reproduce the complete RLHF pipeline (pretraining, reward model, PPO, and its failure mode, reward hacking) and DPO; and a language of additions, on which we reproduce reinforcement learning from verifiable rewards with GRPO. The models are tiny, but the algorithms, the objectives and the phenomena are exactly those of the large ones.

## Language Models as Policies

A language model defines a probability distribution over the next token given the tokens so far, $p_\theta(x_{t+1} \mid x_1, \ldots, x_t)$, and therefore over whole texts, by the chain rule. Used as an agent, the correspondence with the vocabulary of this course is the following:

| Reinforcement learning | Language model |
|---|---|
| state $S_t$ | the prompt $x$ followed by the tokens generated so far, $y_1, \ldots, y_{t-1}$ |
| action $A_t$ | the next token $y_t$, from a vocabulary $\mathcal{V}$ of a few thousand to a few hundred thousand symbols |
| policy $\pi_\theta(a \mid s)$ | the model's next-token distribution $\pi_\theta(y_t \mid x, y_{<t})$ |
| transition | deterministic: the chosen token is appended to the state |
| episode | one response $y = (y_1, \ldots, y_T)$, ending with an end-of-sequence token |
| reward $R$ | a scalar judgement of the complete response, $r(x, y)$, received at the last step; zero before |

The environment is as simple as an environment can be: deterministic, fully observable, with no dynamics to learn. The whole difficulty is in the **reward** and in the **size of the action space**. With a vocabulary of $|\mathcal{V}|$ tokens and responses of $T$ tokens there are $|\mathcal{V}|^T$ possible episodes, the reward is sparse (one number per response) and, in the case of human preferences, is not even given: it must be learned. The probability of a response is the product of the token probabilities,

$\displaystyle \pi_\theta(y \mid x) = \prod_{t=1}^{T} \pi_\theta(y_t \mid x, y_{<t}), \qquad \log \pi_\theta(y \mid x) = \sum_{t=1}^{T} \log \pi_\theta(y_t \mid x, y_{<t})$

and this sum is what every algorithm in this chapter manipulates: the policy gradient of Chapter 08 weights the gradient of each token's log-probability by the return, which here is the single final reward; the importance ratio of PPO is a product of per-token ratios; the KL divergence from a reference model is a sum of per-token divergences.

The training of an assistant proceeds in stages. **Pretraining** fits the next-token distribution on a large corpus by maximum likelihood; this is where the model learns language and facts, and it is supervised learning, not reinforcement learning. **Supervised fine-tuning** (SFT) continues the same objective on a smaller set of demonstrations of the desired behaviour (prompts with good responses written by people), which is behaviour cloning in the sense of Chapter 12. The **reinforcement learning** stage comes last and is the subject of this chapter: it starts from the SFT model, called the **reference model** $\pi_{\text{ref}}$, and improves it with respect to a reward.

## A Toy Language

We need a language small enough that a transformer learns it in a minute, but with enough structure that a response can be good or bad in more than one way. Our toy language has a vocabulary of fourteen words plus three special tokens, and a grammar: a sentence is a determiner, an optional adjective, a noun, a verb, a determiner, an optional adjective, a noun, and a full stop. "the happy cat likes a dog ." is a sentence; "cat cat likes ." is not. The **pretraining corpus** is a set of random sentences drawn from the grammar, which plays the role of the text of the internet, and the model trained on it plays the role of the pretrained language model:

In [ ]:
import numpy as np
import torch
import random
import matplotlib.pyplot as plt

# the vocabulary: special tokens first
PAD, BOS, EOS = 0, 1, 2
WORDS = ["<pad>", "<bos>", "<eos>", "the", "a", "cat", "dog", "bird", "fish",
         "sees", "likes", "chases", "eats", "big", "small", "happy", "sad", "."]
TOKEN = {word: index for index, word in enumerate(WORDS)}

DETERMINERS = ["the", "a"]
ADJECTIVES = ["big", "small", "happy", "sad"]
NOUNS = ["cat", "dog", "bird", "fish"]
VERBS = ["sees", "likes", "chases", "eats"]

vocab_size = len(WORDS)
max_length = 12   # <bos> + up to 8 words (full stop included) + <eos>, with room to spare

In [ ]:
def random_sentence():
    # determiner (adjective)? noun verb determiner (adjective)? noun .
    words = [random.choice(DETERMINERS)]
    if random.random() < 0.5:
        words.append(random.choice(ADJECTIVES))
    words.append(random.choice(NOUNS))
    words.append(random.choice(VERBS))
    words.append(random.choice(DETERMINERS))
    if random.random() < 0.5:
        words.append(random.choice(ADJECTIVES))
    words.append(random.choice(NOUNS))
    words.append(".")
    return words


def encode(words):
    # <bos> words <eos>, padded to the maximum length
    tokens = [BOS] + [TOKEN[w] for w in words] + [EOS]
    return tokens + [PAD] * (max_length - len(tokens))


def decode(tokens):
    # the words between <bos> and <eos> (or the end)
    words = []
    for token in tokens[1:]:
        if token == EOS or token == PAD:
            break
        words.append(WORDS[token])
    return " ".join(words)


random.seed(34)
corpus = torch.as_tensor([encode(random_sentence()) for _ in range(20_000)])

print("Corpus:", corpus.shape)
for i in range(5):
    print("  ", decode(corpus[i].tolist()))

The model is the causal transformer of Chapter 12, with token embeddings in place of the state and return embeddings, a learned positional embedding, and an output layer with one logit per token of the vocabulary. The forward pass returns, at every position, the logits of the **next** token:

In [ ]:
class TinyLanguageModel(torch.nn.Module):

    def __init__(self, vocab_size, max_length, embedding_size=64, n_layers=2, n_heads=4, dropout=0.0):
        super().__init__()

        self.embed_token = torch.nn.Embedding(vocab_size, embedding_size)
        self.embed_position = torch.nn.Embedding(max_length, embedding_size)

        block = torch.nn.TransformerEncoderLayer(
            d_model=embedding_size, nhead=n_heads, dim_feedforward=4 * embedding_size,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = torch.nn.TransformerEncoder(block, num_layers=n_layers)
        self.norm = torch.nn.LayerNorm(embedding_size)
        self.head = torch.nn.Linear(embedding_size, vocab_size)

    def forward(self, tokens):

        # token and position embeddings
        positions = torch.arange(tokens.shape[1]).unsqueeze(0)
        hidden = self.embed_token(tokens) + self.embed_position(positions)

        # causal transformer: each position sees only the tokens before it
        mask = torch.nn.Transformer.generate_square_subsequent_mask(tokens.shape[1])
        hidden = self.norm(self.transformer(hidden, mask=mask, is_causal=True))

        # logits of the next token at every position
        return self.head(hidden)

### Pretraining

Pretraining is maximum likelihood: the loss is the cross-entropy between the predicted next-token distribution and the actual next token, at every position, averaged over the corpus. Padding positions are ignored:

In [ ]:
def pretrain(model, corpus, steps=2000, batch_size=128, learning_rate=0.001):

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

    for step in range(1, steps + 1):

        batch = corpus[torch.randint(0, len(corpus), (batch_size,))]
        inputs, targets = batch[:, :-1], batch[:, 1:]

        # next-token cross-entropy, ignoring the padding
        logits = model(inputs)
        loss = torch.nn.functional.cross_entropy(logits.reshape(-1, vocab_size), targets.reshape(-1), ignore_index=PAD)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 200 == 0:
            print(f"Step {step:05}, loss {loss.item():.4f}", end="\r", flush=True)

    return model


torch.manual_seed(34)
reference_model = pretrain(TinyLanguageModel(vocab_size, max_length), corpus)

### Generation

A response is generated token by token: the model is given the prompt, a token is sampled from its next-token distribution, appended, and the process repeats until the end-of-sequence token or the maximum length. We generate a whole batch at once. The function returns the token sequences and a **mask** marking the positions of the response (from the first generated token to the end-of-sequence token included), which the algorithms below use to pick out the tokens that count as actions:

In [ ]:
def generate(model, prompts, max_new_tokens):

    # prompts: [B, P]; the response is sampled token by token
    tokens = prompts.clone()
    finished = torch.zeros(len(prompts), dtype=torch.bool)
    response_mask = torch.zeros(len(prompts), prompts.shape[1] + max_new_tokens, dtype=torch.bool)

    with torch.no_grad():
        for step in range(max_new_tokens):

            logits = model(tokens)[:, -1]
            logits[:, [PAD, BOS]] = -torch.inf     # the action space is the vocabulary of words plus <eos>
            next_tokens = torch.distributions.Categorical(logits=logits).sample()

            # after the end of the response, pad
            next_tokens[finished] = PAD
            response_mask[:, prompts.shape[1] + step] = ~finished
            finished = finished | (next_tokens == EOS)

            tokens = torch.cat([tokens, next_tokens.unsqueeze(1)], dim=1)

    return tokens, response_mask


def token_distributions(model, tokens):

    # the next-token distribution at every position (position t predicts token t+1); <pad> and <bos> are never generated
    logits = model(tokens[:, :-1])
    logits[:, :, [PAD, BOS]] = -torch.inf
    return torch.distributions.Categorical(logits=logits)


def token_log_probs(model, tokens, with_entropy=False):

    # log-probability of every token given the previous ones, and the entropy of the distribution it was drawn from;
    # the padding positions are set to zero (they are masked out of every loss anyway)
    distributions = token_distributions(model, tokens)
    log_probs = torch.where(tokens[:, 1:] == PAD, 0.0, distributions.log_prob(tokens[:, 1:]))
    return (log_probs, distributions.entropy()) if with_entropy else log_probs

In [ ]:
# the prompt is just the beginning-of-sequence token
bos_prompts = torch.full((8, 1), BOS)

samples, _ = generate(reference_model, bos_prompts, max_new_tokens=max_length - 1)
for sample in samples:
    print("  ", decode(sample.tolist()))

The pretrained model has learned the grammar: it produces sentences of the corpus, with the variety of the corpus. It has no preferences among them, because the corpus had none. We now introduce the preferences.

## Learning a Reward from Human Preferences

### Why a Reward Model

Human preferences about text are not available as a reward function. Nobody can write down a formula for "helpful"; what people *can* do reliably is **compare** two responses and say which one they prefer. RLHF therefore collects a dataset of comparisons, triples $(x, y_w, y_l)$ of a prompt, a preferred ("winning") response and a rejected ("losing") one, and trains a **reward model** $r_\phi(x, y)$ (we reuse $\phi$ for its parameters: there is no critic in this chapter), a network that reads a prompt and a response and outputs a scalar, to be consistent with the comparisons. The reward model then provides the reward for reinforcement learning, as a stand-in for the human annotators, who could never label the millions of responses that policy optimisation needs.

The standard way to turn comparisons into a scalar is the **Bradley-Terry model**: the probability that a human prefers $y_w$ to $y_l$ is the sigmoid of the difference of their scores,

$\displaystyle P(y_w \succ y_l \mid x) = \sigma\big( r_\phi(x, y_w) - r_\phi(x, y_l) \big) = \frac{1}{1 + e^{-(r_\phi(x, y_w) - r_\phi(x, y_l))}}$

so that equal scores give a coin flip and a difference of a few units gives near certainty. The reward model is trained by maximum likelihood on the comparisons, that is by minimising

$\displaystyle \mathcal{L}(\phi) = - \mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}}\Big[ \log \sigma\big( r_\phi(x, y_w) - r_\phi(x, y_l) \big) \Big]$

which is the logistic regression loss on the score difference. Note that only differences matter: the reward model's scale and offset are arbitrary, a point that will return with DPO.

### A Hidden Human

In our toy world the "human" is a hidden scoring function that we write down but never show to the algorithms: it likes sentences about cats and happy sentences, and it does not like text that is not a sentence at all. Every comparison is decided by this function, with some noise, exactly as human annotators decide with some inconsistency. The algorithms will only ever see the comparisons, and we will use the hidden score, at the end, to judge how well they recovered it:

In [ ]:
import re

GRAMMAR = re.compile(r"^(the|a) ((big|small|happy|sad) )?(cat|dog|bird|fish) (sees|likes|chases|eats) (the|a) ((big|small|happy|sad) )?(cat|dog|bird|fish) \.$")


def is_grammatical(words):
    return GRAMMAR.match(words) is not None


def human_score(words):
    # the hidden preference: cats and happiness, but only in proper sentences
    if not is_grammatical(words):
        return 0.0
    return 1.0 * words.split().count("cat") + 0.5 * words.split().count("happy")


def human_prefers(first, second):
    # a noisy comparison: the better sentence wins with a Bradley-Terry probability
    probability = 1.0 / (1.0 + np.exp(-(human_score(first) - human_score(second))))
    return random.random() < probability

The comparison dataset is built the way it is built in practice: responses are sampled from the reference model, paired, and labelled by the (hidden) human:

In [ ]:
def collect_comparisons(model, n_pairs):

    first, _ = generate(model, torch.full((n_pairs, 1), BOS), max_length - 1)
    second, _ = generate(model, torch.full((n_pairs, 1), BOS), max_length - 1)

    winners, losers = [], []
    for a, b in zip(first, second):
        if human_prefers(decode(a.tolist()), decode(b.tolist())):
            winners.append(a); losers.append(b)
        else:
            winners.append(b); losers.append(a)

    return torch.stack(winners), torch.stack(losers)


random.seed(34)
torch.manual_seed(34)

winners, losers = collect_comparisons(reference_model, n_pairs=4000)
test_winners, test_losers = collect_comparisons(reference_model, n_pairs=1000)

for i in range(4):
    print(f"  preferred: {decode(winners[i].tolist()):40s}  rejected: {decode(losers[i].tolist())}")

### Training the Reward Model

The reward model is a transformer of the same kind, with a scalar head instead of the vocabulary head, applied to the hidden state at the **last token** of the response, which, thanks to the causal mask, has seen the whole text. It is trained with the Bradley-Terry loss on the comparisons:

In [ ]:
class RewardModel(torch.nn.Module):

    def __init__(self, vocab_size, max_length, embedding_size=64, n_layers=2, n_heads=4):
        super().__init__()

        self.body = TinyLanguageModel(vocab_size, max_length, embedding_size, n_layers, n_heads)
        self.body.head = torch.nn.Identity()     # keep the hidden states
        self.score = torch.nn.Linear(embedding_size, 1)

    def forward(self, tokens):

        # the hidden state at the last token of each sequence (the end-of-sequence token, or the last one)
        hidden = self.body(tokens)
        lengths = (tokens != PAD).sum(dim=1) - 1
        last = hidden[torch.arange(len(tokens)), lengths]

        return self.score(last).squeeze(-1)


def train_reward_model(winners, losers, steps=1500, batch_size=64, learning_rate=0.0005):

    model = RewardModel(vocab_size, max_length)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

    for step in range(1, steps + 1):

        indices = torch.randint(0, len(winners), (batch_size,))

        # Bradley-Terry: minus log-probability that the preferred response scores higher
        loss = -torch.nn.functional.logsigmoid(model(winners[indices]) - model(losers[indices])).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 100 == 0:
            print(f"Step {step:05}, loss {loss.item():.4f}", end="\r", flush=True)

    return model


torch.manual_seed(34)
reward_model = train_reward_model(winners, losers)

with torch.no_grad():
    accuracy = (reward_model(test_winners) > reward_model(test_losers)).float().mean().item()
print(f"\nReward model accuracy on held-out comparisons: {accuracy:.3f}")

The accuracy on held-out comparisons is well below one, and that is expected: the human's labels are noisy (close scores give nearly random preferences), so no model can agree with them all the time. What matters is whether the reward model has captured the *direction* of the preferences; we can check by scoring a few hand-written sentences, including one that is not a sentence at all:

In [ ]:
def reward_of(words):
    tokens = torch.as_tensor([encode(words.split())])
    with torch.no_grad():
        return reward_model(tokens).item()

for sentence in ["a dog sees a fish .", "the cat sees a fish .", "the happy cat likes the happy cat .", "cat cat cat cat cat cat cat cat cat"]:
    print(f"  {sentence:42s} reward model {reward_of(sentence):6.2f}   hidden human {human_score(sentence):4.1f}")

On proper sentences the reward model ranks as the human does. On the last string, which is not a sentence and which the human scores zero, the reward model gives a high score: it has learned that cats are good, it has never seen text without a grammar, and it **extrapolates**. The reward model is accurate on the distribution of the comparisons, the responses of the reference model, and unreliable outside it. This is the extrapolation problem of Chapter 12 in a new guise, and it is about to matter.

## RLHF with PPO

### The Objective

With a reward model in hand, reinforcement learning can begin. The obvious objective, maximise the expected reward of the responses,

$\displaystyle \max_\theta\ \mathbb{E}_{x \sim \mathcal{D},\ y \sim \pi_\theta(\cdot \mid x)}\big[ r_\phi(x, y) \big]$

is the wrong one, for the reason we just saw: the reward model is only trustworthy near the distribution of the reference model, and an optimiser that is free to move anywhere will find the regions where the reward model is wrong in its favour, which is exactly where its scores are highest. Optimising a learned proxy until it breaks is called **reward hacking**, and it is the central failure mode of RLHF: [Gao, Schulman and Hilton (2023)](./papers/2023%20-%20Scaling%20Laws%20for%20Reward%20Model%20Overoptimization.pdf) measured it systematically, showing that the true reward first rises and then falls as the policy moves away from the reference, with the KL divergence from the reference as the natural measure of that distance. The remedy, introduced with the [first RLHF fine-tuning of language models](./papers/2019%20-%20Fine-Tuning%20Language%20Models%20from%20Human%20Preferences.pdf), is to penalise the policy for straying from the reference model, with the KL divergence of Chapter 11:

$\displaystyle \max_\theta\ \mathbb{E}_{x \sim \mathcal{D},\ y \sim \pi_\theta(\cdot \mid x)}\Big[ r_\phi(x, y) - \beta\, \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)} \Big] = \mathbb{E}_x\Big[ \mathbb{E}_{y \sim \pi_\theta}[r_\phi(x,y)] - \beta\, D_{\text{KL}}\big( \pi_\theta(\cdot \mid x)\ \big\|\ \pi_{\text{ref}}(\cdot \mid x) \big) \Big]$

The coefficient $\beta$ (the KL coefficient; in this chapter $\beta$ no longer denotes the entropy coefficient of Chapters 08 to 11) sets the price of divergence: a small $\beta$ lets the policy chase the reward model, a large one keeps it close to the reference. The KL term does three things at once. It keeps the policy where the reward model is valid; it preserves what the reference model knew, the grammar in our toy world, the knowledge and the fluency in a real one; and it acts as the trust region of Chapter 11, but with respect to the *initial* policy rather than the policy of the previous iteration. Since the log-ratio of a whole response is the sum of the per-token log-ratios, the penalty can be charged **token by token**, as a negative reward at every step,

$\displaystyle R_{t+1} = - \beta \Big( \log \pi_\theta(y_t \mid x, y_{<t}) - \log \pi_{\text{ref}}(y_t \mid x, y_{<t}) \Big) \quad \text{for } t < T, \qquad R_{T+1} = r_\phi(x, y) - \beta \Big( \log \pi_\theta(y_T \mid x, y_{<T}) - \log \pi_{\text{ref}}(y_T \mid x, y_{<T}) \Big)$

(with the convention of the course: $R_{t+1}$ is the reward for the action $A_t = y_t$)

and the problem becomes an ordinary episodic reinforcement learning problem, with dense penalties and a terminal reward, to which PPO applies directly.

### PPO for Sequences

The implementation is the PPO of Chapter 11, with the response tokens as actions. The ratio $r_t(\theta)$ is the per-token probability ratio between the current policy and the policy that generated the batch; the advantage is estimated at the level of the **response**: each response's total reward, the reward model's score minus the KL penalty summed over the tokens, is centred and scaled across the batch, and every token of the response receives the same advantage. (Full-scale implementations also train a critic and use GAE; with a single terminal reward and short responses the sequence-level baseline is simpler and, as we will see with GRPO, is what the most recent methods use.) The clipped surrogate is averaged over the response tokens and optimised for a few epochs on each batch of generated responses, together with the **entropy bonus** of Chapter 11, with a coefficient of $0.1$, ten times the value used there (the symbol $\beta$ is now taken by the KL penalty). The bonus matters more here than it did on Cart-Pole: with a single terminal reward, no critic and a batch-level baseline, a policy that has found a good response is pushed hard towards it, and with a small bonus it collapses within a hundred iterations onto one sentence, after which every sample is identical, every advantage is zero and learning stops (a batch of identical responses carries no information, and the entropy gradient of a nearly deterministic distribution is too small to revive it). The reference model and the reward model are frozen:

In [ ]:
import copy


def response_log_ratio(model, reference, tokens, response_mask):

    # per-token log-probabilities of the response under the policy and the reference
    log_probs = token_log_probs(model, tokens)
    with torch.no_grad():
        reference_log_probs = token_log_probs(reference, tokens)

    mask = response_mask[:, 1:].float()
    return log_probs, reference_log_probs, mask


def rlhf_update(policy, reference, optimizer, tokens, response_mask, rewards, beta=0.1, epochs=2, epsilon=0.2, entropy_coefficient=0.1):

    # log-probabilities of the batch under the policy that generated it (old) and the reference
    with torch.no_grad():
        old_log_probs, reference_log_probs, mask = response_log_ratio(policy, reference, tokens, response_mask)

        # total reward of each response: reward model score minus the KL penalty, summed over its tokens
        kl = ((old_log_probs - reference_log_probs) * mask).sum(dim=1)
        total = rewards - beta * kl

        # one advantage per response, shared by all its tokens
        advantages = (total - total.mean()) / (total.std() + 1e-8)

    for epoch in range(epochs):

        # per-token ratios between the current policy and the old one, and entropy of the current policy
        log_probs, entropy = token_log_probs(policy, tokens, with_entropy=True)
        ratios = (log_probs - old_log_probs).exp()

        # clipped surrogate and entropy bonus, averaged over the response tokens
        unclipped = ratios * advantages.unsqueeze(1)
        clipped = ratios.clamp(1.0 - epsilon, 1.0 + epsilon) * advantages.unsqueeze(1)
        surrogate = (torch.min(unclipped, clipped) * mask).sum() / mask.sum()
        entropy = (entropy * mask).sum() / mask.sum()

        optimizer.zero_grad()
        (-(surrogate + entropy_coefficient * entropy)).backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer.step()

    return kl.mean().item()

To follow what happens during training we measure, on every batch, the reward model's score (what the optimiser sees), the hidden human score (the truth), the fraction of grammatical responses, and the KL divergence from the reference. The training loop alternates generation and updates:

In [ ]:
def evaluate_responses(tokens):
    sentences = [decode(t.tolist()) for t in tokens]
    return (
        float(np.mean([human_score(s) for s in sentences])),
        float(np.mean([is_grammatical(s) for s in sentences])),
    )


def rlhf(reference, reward_model, beta, iterations=500, batch_size=64, learning_rate=0.0003):

    # the policy starts as a copy of the reference model
    policy = copy.deepcopy(reference)
    optimizer = torch.optim.Adam(policy.parameters(), lr=learning_rate)

    log = {"reward": [], "human": [], "grammatical": [], "kl": []}

    for iteration in range(1, iterations + 1):

        # generate a batch of responses with the current policy and score them
        tokens, response_mask = generate(policy, torch.full((batch_size, 1), BOS), max_length - 1)
        with torch.no_grad():
            rewards = reward_model(tokens)

        # one PPO update
        kl = rlhf_update(policy, reference, optimizer, tokens, response_mask, rewards, beta)

        # bookkeeping
        human, grammatical = evaluate_responses(tokens)
        log["reward"].append(rewards.mean().item()); log["human"].append(human)
        log["grammatical"].append(grammatical); log["kl"].append(kl)

        if iteration % 10 == 0:
            print(f"Iteration {iteration:04}, reward model {rewards.mean().item():5.2f}, human {human:5.2f}, grammatical {grammatical:4.2f}, KL {kl:6.2f}", end="\r", flush=True)

    return policy, log

### Reward Hacking

We first run the optimisation **without** the KL penalty, $\beta = 0$, that is, we simply maximise the reward model:

In [ ]:
torch.manual_seed(34)
random.seed(34)

hacked_policy, hacked_log = rlhf(reference_model, reward_model, beta=0.0)

print("\nSamples from the policy trained with beta = 0:")
samples, _ = generate(hacked_policy, bos_prompts, max_length - 1)
for sample in samples:
    print("  ", decode(sample.tolist()))

And then **with** the penalty:

In [ ]:
torch.manual_seed(34)
random.seed(34)

aligned_policy, aligned_log = rlhf(reference_model, reward_model, beta=0.3)

print("\nSamples from the policy trained with beta = 0.3:")
samples, _ = generate(aligned_policy, bos_prompts, max_length - 1)
for sample in samples:
    print("  ", decode(sample.tolist()))

In [ ]:
def plot_rlhf(logs, labels, colors):

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    axes[0].set_title("Reward model score (the proxy)")
    axes[1].set_title("Hidden human score (the truth)")
    axes[2].set_title("KL divergence from the reference")

    for log, label, color in zip(logs, labels, colors):
        axes[0].plot(log["reward"], color=color, label=label)
        axes[1].plot(log["human"], color=color, label=label)
        axes[2].plot(log["kl"], color=color, label=label)

    for axis in axes:
        axis.set_xlabel("Iterations")
        axis.legend()
    plt.show()


plot_rlhf([hacked_log, aligned_log], ["beta = 0", "beta = 0.3"], ["red", "green"])

The two runs tell the story of RLHF in miniature. Without the penalty, the policy first does the right thing: it moves towards sentences about happy cats, and the hidden human score rises together with the reward model's. Then the reward model's score keeps climbing while the human score **collapses to zero**: the policy has found strings that the reward model scores above any genuine sentence (a sentence with extra cats glued in, with its verb removed, or cut off before the full stop: which one depends on the seed), and it has converged onto one of them, with a KL divergence from the reference of tens of nats, the signature of a policy that has left the language. The collapse is typically **sudden**, and its timing varies from seed to seed: the policy climbs through the best genuine sentences, which the reward model also scores highly, until a sample reveals the exploit, after which the whole batch follows within a few iterations. The exploit is found by *exploration*, the residual randomness that the entropy bonus preserves; this is why the bonus is not an innocent ingredient here, and why large-scale practice measures the hidden quantity, with fresh human evaluation of the trained policy, rather than trusting the reward model's curve. The optimiser has done exactly what it was asked; what it was asked was wrong. With the penalty, the policy moves towards sentences about cats, and happy ones, while remaining a policy over *sentences* and keeping some variety: the human score rises and stays up, the KL divergence stabilises at a few nats, and the samples read like the corpus with the preferences applied. The reward model's score is lower than in the hacked run, which is correct: the hacked score was a fiction.

The value of $\beta$ is a genuine hyperparameter of RLHF. Too small, and the run above shows what happens; too large, and the policy barely moves from the reference (with $\beta = 1$ on this task the KL divergence stays below half a nat and the human score improves much less). It is chosen by watching exactly these curves, and in practice also by monitoring the behaviour of the model on tasks it was good at before, which is what the KL term protects. Everything else in the pipeline, the comparisons, the reward model, the PPO machinery, the per-token KL, is as in the large-scale implementations; only the sizes differ.

## Direct Preference Optimization (DPO)

RLHF with PPO has moving parts: a reward model to train, a policy, a reference, often a critic, sampling from the policy at every iteration, and the delicate hyperparameters of PPO. [**Direct Preference Optimization**](./papers/2023%20-%20Direct%20Preference%20Optimization%20-%20Your%20Language%20Model%20is%20Secretly%20a%20Reward%20Model.pdf), introduced by Rafailov and colleagues in 2023, observes that the KL-regularised objective can be optimised **directly on the comparison data**, with a supervised loss and no reward model, no sampling and no reinforcement learning loop. The derivation is short and worth following in full, because it ties together the Bradley-Terry model and the KL-regularised objective.

### The Optimal Policy of the KL-Regularised Objective

Fix a prompt $x$ and consider the objective of RLHF as a function of the distribution $\pi(\cdot \mid x)$ over responses, for a given reward $r$:

$\displaystyle \max_\pi\ \mathbb{E}_{y \sim \pi}\big[ r(x,y) \big] - \beta\, D_{\text{KL}}\big( \pi(\cdot \mid x)\ \big\|\ \pi_{\text{ref}}(\cdot \mid x) \big) = \max_\pi\ \sum_y \pi(y \mid x) \Big[ r(x,y) - \beta \log \frac{\pi(y \mid x)}{\pi_{\text{ref}}(y \mid x)} \Big]$

This problem has a closed-form solution. Define the distribution

$\displaystyle \pi^*(y \mid x) = \frac{1}{Z(x)}\, \pi_{\text{ref}}(y \mid x)\, \exp\Big( \frac{r(x,y)}{\beta} \Big), \qquad Z(x) = \sum_y \pi_{\text{ref}}(y \mid x)\, \exp\Big( \frac{r(x,y)}{\beta} \Big)$

the reference distribution reweighted by the exponentiated reward and normalised. Substituting $r(x,y) = \beta \log \frac{\pi^*(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x)$, which is just this definition solved for the reward, into the objective gives

$\displaystyle \sum_y \pi(y \mid x) \Big[ \beta \log \frac{\pi^*(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x) - \beta \log \frac{\pi(y \mid x)}{\pi_{\text{ref}}(y \mid x)} \Big] = \beta \log Z(x) - \beta\, D_{\text{KL}}\big( \pi(\cdot \mid x)\ \big\|\ \pi^*(\cdot \mid x) \big)$

since the reference cancels inside the logarithm. The first term does not depend on $\pi$ and the KL divergence is minimised, at zero, by $\pi = \pi^*$: **the optimal policy is the reference reweighted by $e^{r/\beta}$**. This is an exact result, and it says what RLHF is trying to compute. It is not directly usable, because $Z(x)$ is a sum over all possible responses, which is why RLHF resorts to policy gradient methods.

### The Reward Hidden in the Policy

DPO turns the relation around. For *any* policy $\pi_\theta$, define the **implicit reward**

$\displaystyle r_\theta(x,y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x)$

This is the reward for which $\pi_\theta$ would be the optimal policy. Now plug this reward into the Bradley-Terry model of the comparisons. The intractable term $\beta \log Z(x)$ is the same for both responses to the same prompt and **cancels** in the difference:

$\displaystyle P\big( y_w \succ y_l \mid x \big) = \sigma\big( r_\theta(x, y_w) - r_\theta(x, y_l) \big) = \sigma\Big( \beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \Big)$

Maximising the likelihood of the comparisons under this model is the **DPO loss**:

$\displaystyle \mathcal{L}_{\text{DPO}}(\theta) = - \mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}}\Big[ \log \sigma\Big( \beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \Big) \Big]$

It is the reward model's loss, with the reward model replaced by the policy's own log-ratios: the language model is, in the words of the paper's title, secretly a reward model. Training on this loss fits the policy whose implicit reward explains the comparisons best, which is the policy that RLHF would converge to for the reward model that fits the comparisons best. No reward model is trained, nothing is sampled, and the whole procedure is a supervised fine-tuning on the comparison dataset, with the reference model used only to compute the log-ratios.

The gradient of the loss is also instructive. Writing $\hat r_\theta$ for the implicit reward without the $Z$ term,

$\displaystyle \nabla_\theta \mathcal{L}_{\text{DPO}} = - \beta\, \mathbb{E}\Big[ \underbrace{\sigma\big( \hat r_\theta(x, y_l) - \hat r_\theta(x, y_w) \big)}_{\text{weight}} \Big( \nabla_\theta \log \pi_\theta(y_w \mid x) - \nabla_\theta \log \pi_\theta(y_l \mid x) \Big) \Big]$

the update increases the log-probability of the preferred response and decreases that of the rejected one, like a contrastive version of behaviour cloning, **weighted** by how wrongly the implicit reward currently ranks the pair: pairs that the policy already gets right contribute little, pairs it gets wrong contribute a lot. The coefficient $\beta$ plays the same role as in RLHF, that of the strength of the attachment to the reference. The implementation needs the sequence log-probabilities of the winning and losing responses under the policy and the reference:

In [ ]:
def sequence_log_prob(model, tokens):

    # log-probability of the whole response (tokens after <bos>, up to <eos>)
    log_probs = token_log_probs(model, tokens)
    mask = (tokens[:, 1:] != PAD).float()
    return (log_probs * mask).sum(dim=1)


def dpo(reference, winners, losers, beta=0.1, steps=600, batch_size=64, learning_rate=0.00005):

    policy = copy.deepcopy(reference)
    optimizer = torch.optim.Adam(policy.parameters(), lr=learning_rate)

    # the reference log-probabilities of the whole dataset do not change: compute them once
    with torch.no_grad():
        reference_winners = sequence_log_prob(reference, winners)
        reference_losers = sequence_log_prob(reference, losers)

    for step in range(1, steps + 1):

        indices = torch.randint(0, len(winners), (batch_size,))

        # implicit rewards (without the constant Z, which cancels)
        reward_winners = beta * (sequence_log_prob(policy, winners[indices]) - reference_winners[indices])
        reward_losers = beta * (sequence_log_prob(policy, losers[indices]) - reference_losers[indices])

        # the DPO loss: Bradley-Terry on the implicit rewards
        loss = -torch.nn.functional.logsigmoid(reward_winners - reward_losers).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 50 == 0:
            print(f"Step {step:05}, loss {loss.item():.4f}", end="\r", flush=True)

    return policy


torch.manual_seed(34)
dpo_policy = dpo(reference_model, winners, losers)

print("\nSamples from the DPO policy:")
samples, _ = generate(dpo_policy, bos_prompts, max_length - 1)
for sample in samples:
    print("  ", decode(sample.tolist()))

We compare the reference, the two RLHF policies and the DPO policy on the same metrics, over a large sample of responses:

In [ ]:
def assess(model, n=1000):
    tokens, response_mask = generate(model, torch.full((n, 1), BOS), max_length - 1)
    human, grammatical = evaluate_responses(tokens)
    with torch.no_grad():
        reward = reward_model(tokens).mean().item()
        log_probs, reference_log_probs, mask = response_log_ratio(model, reference_model, tokens, response_mask)
        kl = ((log_probs - reference_log_probs) * mask).sum(dim=1).mean().item()
    return reward, human, grammatical, kl


print(f"{'policy':22s} {'reward model':>13s} {'human score':>12s} {'grammatical':>12s} {'KL':>8s}")
for name, model in (("reference", reference_model), ("RLHF, beta = 0", hacked_policy), ("RLHF, beta = 0.3", aligned_policy), ("DPO, beta = 0.1", dpo_policy)):
    reward, human, grammatical, kl = assess(model)
    print(f"{name:22s} {reward:13.2f} {human:12.2f} {grammatical:12.2f} {kl:8.2f}")

DPO reaches a human score comparable to that of the KL-regularised RLHF policy, with grammatical responses and a moderate divergence from the reference, from the comparison data alone and in a fraction of the computation. Its simplicity has made it, and its many variants, the most common way of training on preference data. It has limitations of its own. It is an **offline** method in the sense of Chapter 12: it learns only from the responses in the comparison dataset, which were generated by the reference model, so it cannot discover behaviours that the dataset does not contain, while RLHF samples from the current policy and can; and its implicit reward, like any reward model, is only reliable near the data, so that training DPO for too long, or with too small a $\beta$, degrades the policy in ways that resemble reward hacking. In practice the two approaches coexist, and the strongest models have used both.

## Verifiable Rewards and GRPO

Human preferences are not the only source of reward. For problems with a **checkable answer**, arithmetic, mathematics, programming with test cases, logical puzzles, the reward can be computed by a program: $1$ if the answer is correct, $0$ otherwise. No reward model is needed, there is much less to hack (the verifier is exact on what it checks, although a policy can still exploit what it does not check, such as the format of the answer), and the policy can be optimised for much longer. This is **reinforcement learning from verifiable rewards**, and it is the method that produced the recent generation of models that reason step by step before answering: the model learns to generate whatever intermediate text increases the probability that the final answer is correct, and, given enough optimisation, that text turns into reasoning.

The algorithm commonly used is [**Group Relative Policy Optimization**](./papers/2024%20-%20DeepSeekMath%20-%20Pushing%20the%20Limits%20of%20Mathematical%20Reasoning%20in%20Open%20Language%20Models.pdf) (GRPO), introduced by the DeepSeek team in 2024. It is PPO stripped of its critic. For each prompt $x$ the policy generates a **group** of $G$ responses $y_1, \ldots, y_G$, which are scored by the verifier, and the advantage of each response is its reward standardised **within the group**:

$\displaystyle \hat A_i = \frac{ r_i - \text{mean}(r_1, \ldots, r_G) }{ \text{std}(r_1, \ldots, r_G) }$

The group mean is a Monte Carlo estimate of the value of the prompt, $v(x)$, which is exactly what a critic would have provided: responses better than the average for *their* prompt get a positive advantage, worse ones a negative one, and a prompt on which all responses are equally good or bad contributes nothing, which is right, since there is nothing to learn from it. This is the baseline of Chapter 08, computed from samples instead of learned, and it removes the critic, which for a language model would be a second network as large as the policy. The policy is then updated with the clipped surrogate of PPO, with every token of a response receiving its response's advantage, as in our RLHF implementation, plus the KL penalty with respect to the reference, which GRPO adds to the loss directly rather than to the reward, using the per-token estimator

$\displaystyle \hat D_{\text{KL}, t} = \frac{\pi_{\text{ref}}(y_t \mid x, y_{<t})}{\pi_\theta(y_t \mid x, y_{<t})} - \log \frac{\pi_{\text{ref}}(y_t \mid x, y_{<t})}{\pi_\theta(y_t \mid x, y_{<t})} - 1$

which, like the estimator of Chapter 11 (the $k_3$ estimator of [Schulman's note on approximating KL divergences](http://joschu.net/blog/kl-approx.html)), is non-negative and unbiased for the *value* of the KL divergence when the tokens are sampled from $\pi_\theta$; its gradient, however, is not that of $D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}})$ but that of the reversed divergence $D_{\text{KL}}(\pi_{\text{ref}} \| \pi_\theta)$, which vanishes at the same point and serves the same purpose. The complete GRPO objective, for a group of responses to one prompt, with $\rho_{i,t}(\theta)$ the per-token probability ratio of PPO (renamed to avoid a clash with the rewards $r_i$), is

$\displaystyle \mathcal{L}_{\text{GRPO}}(\theta) = - \frac{1}{G} \sum_{i=1}^{G} \frac{1}{T_i} \sum_{t=1}^{T_i} \Big[ \min\Big( \rho_{i,t}(\theta)\, \hat A_i,\ \text{clip}\big( \rho_{i,t}(\theta), 1-\epsilon, 1+\epsilon \big)\, \hat A_i \Big) - \beta\, \hat D_{\text{KL}, i, t} \Big]$

### A Toy Arithmetic Task

Our second toy world is single-digit addition. The vocabulary is the ten digits, the symbols $+$ and $=$, and the special tokens; a text is "a + b = c". The pretraining corpus is **noisy**: the answer is correct in $60\%$ of the examples and a random wrong number in the remaining $40\%$, as if the model had been trained on text containing both correct and incorrect arithmetic, which is what a corpus of the internet does contain. The pretrained model will therefore answer correctly about $60\%$ of the time when sampling. The prompt is "a + b =" and the response is the answer; the verifier checks it:

In [ ]:
DIGITS = [str(d) for d in range(10)]
MATH_WORDS = ["<pad>", "<bos>", "<eos>"] + DIGITS + ["+", "="]
MATH_TOKEN = {word: index for index, word in enumerate(MATH_WORDS)}

math_vocab_size = len(MATH_WORDS)
math_max_length = 9      # <bos> a + b = c c <eos> is 8 tokens; one position spare
prompt_length = 5        # <bos> a + b =


def math_prompt(a, b):
    return [BOS, MATH_TOKEN[str(a)], MATH_TOKEN["+"], MATH_TOKEN[str(b)], MATH_TOKEN["="]]


def math_example(a, b, correct):
    answer = a + b if correct else random.choice([n for n in range(19) if n != a + b])
    tokens = math_prompt(a, b) + [MATH_TOKEN[d] for d in str(answer)] + [EOS]
    return tokens + [PAD] * (math_max_length - len(tokens))


def math_decode(tokens):
    words = []
    for token in tokens[1:]:
        if token == EOS or token == PAD:
            break
        words.append(MATH_WORDS[token])
    return " ".join(words)


def verify(tokens):
    # the reward: 1 if the answer after "=" is the correct sum, 0 otherwise
    words = math_decode(tokens).split()
    try:
        a, b = int(words[0]), int(words[2])
        answer = int("".join(words[4:]))
        return float(answer == a + b)
    except (ValueError, IndexError):
        return 0.0


random.seed(34)
math_corpus = torch.as_tensor([math_example(random.randint(0, 9), random.randint(0, 9), random.random() < 0.6) for _ in range(20_000)])

for i in range(5):
    print("  ", math_decode(math_corpus[i].tolist()), "   correct" if verify(math_corpus[i].tolist()) else "   wrong")

In [ ]:
def pretrain_math(model, corpus, steps=2000, batch_size=128, learning_rate=0.001):

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

    for step in range(1, steps + 1):

        batch = corpus[torch.randint(0, len(corpus), (batch_size,))]
        inputs, targets = batch[:, :-1], batch[:, 1:]

        logits = model(inputs)
        loss = torch.nn.functional.cross_entropy(logits.reshape(-1, math_vocab_size), targets.reshape(-1), ignore_index=PAD)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 200 == 0:
            print(f"Step {step:05}, loss {loss.item():.4f}", end="\r", flush=True)

    return model


torch.manual_seed(34)
math_reference = pretrain_math(TinyLanguageModel(math_vocab_size, math_max_length), math_corpus)

We measure the accuracy of a model over all the hundred additions, both when sampling an answer and when taking the most probable one at each step (greedy decoding):

In [ ]:
def all_prompts():
    return torch.as_tensor([math_prompt(a, b) for a in range(10) for b in range(10)])


def generate_greedy(model, prompts, max_new_tokens):
    tokens = prompts.clone()
    with torch.no_grad():
        for step in range(max_new_tokens):
            logits = model(tokens)[:, -1]
            logits[:, [PAD, BOS]] = -torch.inf
            next_tokens = logits.argmax(dim=-1)
            tokens = torch.cat([tokens, next_tokens.unsqueeze(1)], dim=1)
    return tokens


def math_accuracy(model):
    prompts = all_prompts()
    sampled, _ = generate(model, prompts, math_max_length - prompt_length)
    greedy = generate_greedy(model, prompts, math_max_length - prompt_length)
    return (
        float(np.mean([verify(t.tolist()) for t in sampled])),
        float(np.mean([verify(t.tolist()) for t in greedy])),
    )


reference_sampled_accuracy, reference_greedy_accuracy = math_accuracy(math_reference)
print(f"Pretrained model: accuracy {reference_sampled_accuracy:.2f} when sampling, {reference_greedy_accuracy:.2f} with greedy decoding")

### GRPO

The implementation follows the objective. For each prompt in a batch, $G$ responses are sampled; the verifier scores them; the advantages are standardised within each group; and the policy is updated with the clipped surrogate and the KL estimator, averaged over the response tokens:

In [ ]:
def grpo_update(policy, reference, optimizer, tokens, response_mask, advantages, beta=0.04, epochs=2, epsilon=0.2):

    with torch.no_grad():
        old_log_probs = token_log_probs(policy, tokens)
        reference_log_probs = token_log_probs(reference, tokens)
    mask = response_mask[:, 1:].float()

    for epoch in range(epochs):

        log_probs = token_log_probs(policy, tokens)

        # clipped surrogate with the group-relative advantage of each response
        ratios = (log_probs - old_log_probs).exp()
        unclipped = ratios * advantages.unsqueeze(1)
        clipped = ratios.clamp(1.0 - epsilon, 1.0 + epsilon) * advantages.unsqueeze(1)
        surrogate = torch.min(unclipped, clipped)

        # per-token KL estimator with respect to the reference
        log_ratio = reference_log_probs - log_probs
        kl = log_ratio.exp() - log_ratio - 1.0

        # average over the response tokens of each response, then over responses
        per_token = (surrogate - beta * kl) * mask
        loss = -(per_token.sum(dim=1) / mask.sum(dim=1)).mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer.step()


def grpo(reference, iterations=100, prompts_per_batch=16, group_size=8, beta=0.04, learning_rate=0.0001):

    policy = copy.deepcopy(reference)
    optimizer = torch.optim.Adam(policy.parameters(), lr=learning_rate)

    log = {"reward": [], "accuracy": []}

    for iteration in range(1, iterations + 1):

        # a batch of prompts, each repeated G times
        pairs = [(random.randint(0, 9), random.randint(0, 9)) for _ in range(prompts_per_batch)]
        prompts = torch.as_tensor([math_prompt(a, b) for a, b in pairs for _ in range(group_size)])

        # a group of responses per prompt, scored by the verifier
        tokens, response_mask = generate(policy, prompts, math_max_length - prompt_length)
        rewards = torch.as_tensor([verify(t.tolist()) for t in tokens]).reshape(prompts_per_batch, group_size)

        # group-relative advantages
        advantages = (rewards - rewards.mean(dim=1, keepdim=True)) / (rewards.std(dim=1, keepdim=True) + 1e-4)
        advantages = advantages.reshape(-1)

        grpo_update(policy, reference, optimizer, tokens, response_mask, advantages, beta)

        log["reward"].append(rewards.mean().item())
        if iteration % 5 == 0:
            sampled_accuracy, _ = math_accuracy(policy)
            log["accuracy"].append(sampled_accuracy)
            print(f"Iteration {iteration:04}, batch reward {rewards.mean().item():.2f}, accuracy {sampled_accuracy:.2f}", end="\r", flush=True)

    return policy, log

In [ ]:
torch.manual_seed(34)
random.seed(34)

math_policy, grpo_log = grpo(math_reference)

sampled_accuracy, greedy_accuracy = math_accuracy(math_policy)
print(f"\nGRPO model: accuracy {sampled_accuracy:.2f} when sampling, {greedy_accuracy:.2f} with greedy decoding")

In [ ]:
plt.figure(figsize=(12, 5))
plt.title("GRPO on the arithmetic task")
plt.xlabel("Iterations")
plt.ylabel("Accuracy when sampling")
plt.plot((np.arange(len(grpo_log["accuracy"])) + 1) * 5, grpo_log["accuracy"], color="purple", linewidth=2, label="GRPO policy")
plt.axhline(reference_sampled_accuracy, color="gray", linestyle=":", linewidth=1, label="pretrained model, sampling")
plt.axhline(0.6, color="gray", linestyle="--", linewidth=1, label="fraction of correct answers in the corpus")
plt.legend()
plt.show()

The accuracy when sampling climbs from about one half to above $0.9$, and it climbs for a reason worth stating precisely. Note what the pretrained model could already do: with **greedy** decoding, taking the most probable token at every step, it answers every addition correctly, because the correct answer was the single most frequent continuation of every prompt in the corpus; what sampling reproduced was the *distribution* of the corpus, correct answers and mistakes alike. The reward does not teach the model arithmetic: it teaches it to **put its probability mass on the continuation that the verifier accepts**, which was already the most likely one. This sharpening of a distribution that already contained the answer is a large part of what reinforcement learning from verifiable rewards does to language models, and it explains both its power, since a capable base model can be made far more reliable with little data, and a frequently observed limitation, since the diversity of the model's outputs decreases as it concentrates on what is rewarded. The question of whether such training can also produce genuinely new capabilities, rather than sharpen existing ones, is one of the open questions of the field.

## Limitations and Outlook

Reinforcement learning for language models is the most visible application of the ideas of this course, and also the one in which their limitations are most visible. The **reward** is the weakest link. A reward model is a function approximator trained on limited data, and a policy optimised against it will find its mistakes: the hacked run of this chapter is a caricature, but the phenomena it caricatures, models that learn to be verbose because annotators liked longer answers, to agree with the user because agreement was rated well, or to sound confident regardless of correctness, are real, and the KL penalty mitigates them without removing them. Verifiable rewards avoid the problem where they exist, but only a fraction of what we want from a model can be verified by a program.

The **data** are the second limit. Comparisons are expensive and noisy, and they encode the preferences of whoever produced them; DPO and its relatives make the most of a fixed dataset but cannot go beyond it, while RLHF can explore but only within the reach of the reward model. The **optimisation** itself inherits the fragilities of the methods we studied: the step-size problem of Chapter 11, which the KL penalty to the reference model and the clipping of PPO and GRPO jointly contain; the loss of diversity that any policy gradient method causes as it concentrates on rewarded behaviour; and the sheer cost of sampling from a model with billions of parameters at every iteration.

What this chapter should leave behind is a way of seeing. A language model is a policy over tokens; fine-tuning it with feedback is an episodic reinforcement learning problem with a learned or verified terminal reward; the algorithms are those of Chapters 08 to 11, with a penalty that keeps the policy near its starting point; and the difficulties are those of Chapter 12, because the reward is only trustworthy where the data were. The vocabulary of this course, value, advantage, policy gradient, trust region, extrapolation, is exactly the vocabulary needed to read the papers on which today's models are built.